# Backtesting the quoting policies — passive-fill replay

Everything in the four preceding notebooks computes *what to quote*. None of them say whether quoting that way **makes money**. This notebook closes that loop.

It replays a recorded trade tape, decides which of the dealer's resting quotes would have been hit, and decomposes the result into exactly the two terms the paper's objective is written over — `Z_T` (spread capture) and `I_T` (inventory value).

**How fills are inferred.** Deribit's tape reports each trade's aggressor `direction`, which is enough to test a resting quote without an order-book replay:

- `direction == 'buy'` — the aggressor lifted an ask, so a resting ask **at or below** the trade price would have been taken. The dealer sells.
- `direction == 'sell'` — the aggressor hit a bid, so a resting bid **at or above** the trade price would have been taken. The dealer buys.

Quotes are always set from the **previous** trade's mark, never the current one, so a quote can never be informed by the trade that fills it. Deribit's mark moves with the tape, so quoting off the current trade's mark would leak the answer into the decision.

**Read the results as directional, not precise.** Two limitations pull in opposite directions and neither is small:
- The replay can only re-quote at *trade timestamps*. A real maker re-quotes on every book update (milliseconds). This **overstates** adverse selection.
- `queue_factor=1.0` assumes you are always at the front of the queue. This **understates** difficulty.

Code lives in `src/market_making/backtest.py`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('../src'))
sys.path.insert(0, os.path.abspath('../src/vol_surface'))

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from market_making import backtest, calibration, quoting

%matplotlib inline
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

# Categorical slots 1-3 of the validated reference palette, in fixed order.
# These three validate all-pairs in both light and dark modes; every chart below
# is paired with a table, which also covers the relief rule for aqua's contrast
# on a light surface. Colors are never cycled or reassigned by rank.
SERIES = ['#2a78d6', '#eb6834', '#1baf7a']
INK, MUTED, GRID = '#0b0b0b', '#52514e', '#dcdcd8'

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': GRID, 'axes.labelcolor': MUTED,
    'axes.titlecolor': INK, 'axes.titlesize': 11, 'axes.titleweight': 'semibold',
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'grid.color': GRID, 'grid.linewidth': 0.6,
    'legend.frameon': False, 'lines.linewidth': 2.0,
})

## 1. Load the tape and calibrate

BTC-PERPETUAL is the only instrument with enough prints for a same-session replay — the whole option chain pooled produces a few hundred trades a day, while the perpetual alone prints ~1000 in under an hour. It is also USD-quoted, so `price_in_underlying_units=False` throughout.

In [ ]:
trades = calibration.fetch_trades('BTC-PERPETUAL', count=1000)
fit = calibration.calibrate_intensity(
    trades, min_trades_per_side=20, n_bins=15, price_in_underlying_units=False
)

S0 = float(trades['mark_price'].iloc[-1])
intensity = fit.bid.intensity.to_absolute(S0)
eps_star = quoting.optimal_premium(intensity)

tagged = (calibration.add_fill_distance(trades, price_in_underlying_units=False)
          .sort_values('timestamp').reset_index(drop=True))
mid = tagged['mid_dollar'].to_numpy()
window_min = (trades['timestamp'].max() - trades['timestamp'].min()) / 1000 / 60

print(f"tape:            {len(trades)} trades over {window_min:.1f} minutes")
print(f"mid drift:       {mid[0]:,.1f} -> {mid[-1]:,.1f}  ({mid[-1] - mid[0]:+,.1f})")
print(f"calibrated C/D:  ${intensity.max_premium:.2f}   eps* = C/2D = ${eps_star:.2f}")
print(f"real touch:      ~$0.50 top-of-book spread, for reference")

## 2. Baselines

Three inventory-blind policies at different widths. `constant_spread_policy` is not an arbitrary strawman — it is exactly Theorem 1's answer, and exactly the `gamma = 0` case that Theorems 2–5 all collapse to. Beating it is the minimum bar for the tilting machinery earning its complexity.

In [ ]:
half_spreads = {
    'model eps* = C/2D': eps_star,
    'half of eps*':      eps_star / 2,
    'at ~real touch':    0.25,
}
policies = {name: backtest.constant_spread_policy(h) for name, h in half_spreads.items()}

table = backtest.compare_policies(trades, policies, price_in_underlying_units=False)
table[['total_pnl', 'spread_pnl', 'inventory_pnl', 'n_fills', 'max_abs_inventory']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
names = list(table.index)
y = np.arange(len(names))

# Two panels, not one with a dual axis: spread P&L and inventory P&L differ by ~30x
# in magnitude, so sharing a linear scale would render spread capture invisible.
for ax, col, color, title in [
    (axes[0], 'spread_pnl',    SERIES[0], 'Spread capture (Z_T)'),
    (axes[1], 'inventory_pnl', SERIES[1], 'Inventory P&L (I_T)'),
]:
    vals = table[col].to_numpy()
    ax.barh(y, vals, color=color, height=0.6)
    ax.axvline(0, color=GRID, lw=1)
    ax.set_title(title)
    ax.set_xlabel('USD')
    for yi, v in zip(y, vals):  # direct labels: values as ink, never the series color
        ax.text(v, yi, f'  {v:,.0f}', va='center',
                ha='left' if v >= 0 else 'right', color=INK, fontsize=9)
    ax.margins(x=0.25)

axes[0].set_yticks(y); axes[0].set_yticklabels(names)
axes[1].set_yticks([])  # shared rows -- label them once, no orphan ticks

fig.suptitle('Spread capture works; inventory destroys it', fontsize=12, color=INK, y=1.02)
plt.tight_layout()

## 3. Does the calibrated intensity predict our own fills?

This is a direct falsification test of the model, independent of P&L: if `lambda(eps)` does not predict the fill rate actually achieved when quoting at `eps`, then the premium optimized on top of it is unreliable no matter what the P&L says.

In [ ]:
rows = []
for name, h in half_spreads.items():
    r = backtest.simulate_fills(trades, policies[name], price_in_underlying_units=False)
    rows.append({'policy': name, 'half_spread': h,
                 'predicted': float(intensity.rate(h)), 'realized': r.realized_fill_rate})
rates = pd.DataFrame(rows).set_index('policy')
rates['ratio'] = rates['realized'] / rates['predicted']
rates

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
x = np.arange(len(rates)); w = 0.38

ax.bar(x - w/2, rates['predicted'], w, label='predicted by lambda(eps)', color=SERIES[0])
ax.bar(x + w/2, rates['realized'],  w, label='realized in replay',       color=SERIES[1])

ax.set_xticks(x); ax.set_xticklabels(rates.index)
ax.set_ylabel('fills / second')
# stated from the data, not hardcoded -- this runs on a live tape and the ratio moves
ax.set_title(f"Calibrated intensity underestimates realized fills by "
             f"{rates['ratio'].min():.1f}-{rates['ratio'].max():.1f}x")
ax.yaxis.grid(True); ax.set_axisbelow(True)
ax.legend()
for xi, (p, r) in enumerate(zip(rates['predicted'], rates['realized'])):
    ax.text(xi + w/2, r, f'{r/p:.1f}x', ha='center', va='bottom', color=INK, fontsize=9)
plt.tight_layout()

## 4. Why: adverse selection

The gap in Section 3 is not noise, and it is not queue position. It is a **mismatch in what the two quantities measure**:

- `calibration.calibrate_intensity` measures distance from the **contemporaneous** mark — how far from fair value aggressive trades happen.
- A resting quote experiences distance from a **stale** mark — the one that was current when the quote was placed.

When the market moves between those two moments, a quote that looked like it sat `eps` away is crossed anyway. The difference between the two distances *is* the adverse selection. Measure it directly: where does the mid go in the trades immediately after each of our fills?

In [ ]:
r_model = backtest.simulate_fills(
    trades, policies['model eps* = C/2D'], price_in_underlying_units=False
)
ts_all = tagged['timestamp'].to_numpy()
HORIZON = 20  # trades ahead

moves = []
for _, row in r_model.fills.iterrows():
    i = int(np.searchsorted(ts_all, row['timestamp']))
    j = min(i + HORIZON, len(mid) - 1)
    # sign it so positive always means "the mid moved in our favour"
    moves.append((mid[j] - mid[i]) * (1.0 if row['side'] == 'bid' else -1.0))
moves = np.array(moves)

print(f"fills analysed:                        {len(moves)}")
print(f"mean post-fill mid move in our favour: ${moves.mean():+.3f}")
print(f"share of fills that moved against us:  {(moves < 0).mean():.0%}")
print(f"edge captured per fill:                ${eps_star:+.3f}")
print(f"net per fill:                          ${eps_star + moves.mean():+.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.hist(moves, bins=40, color=SERIES[0], edgecolor='white', linewidth=0.5)

ax.axvline(0, color=GRID, lw=1.5)
ax.axvline(moves.mean(), color=SERIES[1], lw=2, label=f'mean ${moves.mean():+.2f}')
ax.axvline(eps_star, color=SERIES[2], lw=2, ls='--', label=f'edge captured ${eps_star:+.2f}')

ax.set_xlabel(f'mid move in our favour, {HORIZON} trades after the fill (USD)')
ax.set_ylabel('fills')
ax.set_title('Fills are systematically picked off: the loss dwarfs the edge')
ax.yaxis.grid(True); ax.set_axisbelow(True)
ax.legend()
plt.tight_layout()

## 5. The A/B that matters: does inventory tilting help?

Section 2 let inventory run unbounded, which no real desk would. Re-run with a hard position cap (a risk stop the theorems don't model) and compare a flat spread against quotes that tilt with inventory — the mechanism Theorems 2–5 exist to provide.

In [ ]:
CAP = 5.0

def tilted_policy(slope):
    def policy(mid_now, inventory):
        ask = float(np.clip(eps_star - slope * inventory, 0.0, intensity.max_premium))
        bid = float(np.clip(eps_star + slope * inventory, 0.0, intensity.max_premium))
        return backtest.Quote(bid=mid_now - bid, ask=mid_now + ask)
    return policy

capped = {
    'flat (gamma = 0)': backtest.constant_spread_policy(eps_star),
    'tilted (weak)':    tilted_policy(0.05),
    'tilted (strong)':  tilted_policy(0.30),
}
capped_table = backtest.compare_policies(
    trades, capped, price_in_underlying_units=False, max_abs_position=CAP
)
capped_table[['total_pnl', 'spread_pnl', 'inventory_pnl', 'n_fills', 'final_inventory']]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
paths = {}
for (name, policy), color in zip(capped.items(), SERIES):
    res = backtest.simulate_fills(trades, policy, price_in_underlying_units=False,
                                   max_abs_position=CAP)
    paths[name] = res.inventory_path
    ax.plot(np.arange(len(paths[name])), paths[name]['inventory'], color=color, label=name)

ax.axhline(0, color=GRID, lw=1)
for lim in (CAP, -CAP):
    ax.axhline(lim, color=GRID, lw=1, ls=':')
ax.set_xlabel('trade index'); ax.set_ylabel('inventory (contracts)')

flat_at_cap = (paths['flat (gamma = 0)']['inventory'].abs() >= CAP).mean()
ax.set_title(f'Inventory sits at a cap {flat_at_cap:.0%} of the time; tilt barely moves the path')
ax.yaxis.grid(True); ax.set_axisbelow(True)
ax.legend()
plt.tight_layout()

## What this run actually shows

**1. Spread capture works exactly as designed; inventory P&L destroys it.** Every policy captured positive `Z_T` and lost roughly 30x more on `I_T`. This is the failure mode Theorems 2-5 exist to address -- and the reason the paper writes its objective over both terms rather than maximizing revenue alone.

**2. The intensity model is misspecified for this use.** Realized fill rates ran several-fold above the calibrated `lambda(eps)` (3-6x depending on the tape). Not queue position, not noise -- `calibrate_intensity` measures distance from the *contemporaneous* mark, while a resting quote experiences distance from a *stale* one. Any premium optimized against the former is solving the wrong problem. **This is the highest-value thing to fix next.**

**3. Wider was better.** The model's much-criticised "5-7x too wide" spread outperformed quoting at the real touch, because width is partly protective against being picked off. The earlier concern about that mismatch was less damning than it looked.

**4. Tilting did not rescue it here.** With a hard cap the inventory spends most of the tape pinned at one bound or the other, and tilting mostly just fills more and loses slightly more. This is *not* evidence the paper is wrong -- it is evidence that an hour of one instrument is a single regime, and that re-quote cadence (the caveat up top) dominates everything else on a fast instrument. The paper's tilting is designed for a dealer who re-quotes continuously, not once per trade print.

**Note on reproducibility:** every number above comes from a live tape fetched at run time, so re-running this notebook will not reproduce the exact figures. The signs and orders of magnitude have been stable across runs; the precise values are not.

## What would make these numbers trustworthy

- **Record a multi-day tape.** `backtest.append_trade_log(path, instruments)` de-duplicates on `trade_id`; run it on a schedule. Deribit serves no queryable history, so nothing longer than the last few hours can be tested until the log accumulates. One trending hour is an anecdote, not a sample.
- **Recalibrate `lambda` on stale-mark distance** so the intensity measures what resting quotes actually experience (finding 2).
- **Walk-forward calibration** -- fit on a trailing window, quote the next one. The fit here uses the same window it is evaluated on, which is lookahead bias.
- **Run the whole thing at several `queue_factor` values** and report the range; the gap is the part of the P&L that is really just queue luck.
- **Test on options, not just the perpetual**, once the log is deep enough -- the perpetual is a fast, trending, momentum-dominated instrument and is close to the hardest possible case for a passive maker.